# GraphRAG Legal – Test Notebook

Chạy thử end-to-end Qdrant + Neo4j + Gemini.

In [1]:
import sys
print(sys.executable)


d:\PYTHON\ChatBotRAG\mynenv\Scripts\python.exe


In [2]:
import os, sys, pathlib
from dotenv import load_dotenv
load_dotenv()
ROOT = pathlib.Path().resolve()
if (ROOT/'backend').exists(): sys.path.insert(0, str(ROOT/'backend'))
print('Ready')

Ready


In [3]:
import nest_asyncio, asyncio
nest_asyncio.apply()

In [4]:
from app.core.neo4j import get_driver, get_db
async def check():
    d=get_driver()
    async with d.session(**get_db()) as s:
        r=await s.run('MATCH (n) RETURN count(n) AS c')
        print('Neo4j nodes:', (await r.single())['c'])
await check()

Neo4j nodes: 18420


In [5]:
from app.services.qdrant_service import qdrant_legal_service
from app.services.reranker import reranker_service
query='mức lương tối thiểu ở phường Bến Thành thuộc thành phố Hồ Chí Minh là bao nhiêu?'
h=await qdrant_legal_service.hybrid_search(query, top_k=10)
h2=await reranker_service.rerank(query,h,top_k=5)
print(len(h),'->',len(h2))

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


10 -> 5


In [6]:
from app.agents.researcher import ResearcherAgent
r = ResearcherAgent()
ctx = await r.gather_all_evidence(query)
print('Context:', len(ctx))
for c in ctx[:3]:
    print(c['source'], c['score'])

Context: 10
NGHỊ ĐỊNH QUY ĐỊNH MỨC LƯƠNG TỐI THIỂU ĐỐI VỚI NGƯỜI LAO ĐỘNG LÀM VIỆC THEO HỢP ĐỒNG LAO ĐỘNG - Điều 5 0.3
NGHỊ ĐỊNH QUY ĐỊNH MỨC LƯƠNG TỐI THIỂU ĐỐI VỚI NGƯỜI LAO ĐỘNG LÀM VIỆC THEO HỢP ĐỒNG LAO ĐỘNG - Điều 5 0.15
NGHỊ ĐỊNH QUY ĐỊNH MỨC LƯƠNG TỐI THIỂU ĐỐI VỚI NGƯỜI LAO ĐỘNG LÀM VIỆC THEO HỢP ĐỒNG LAO ĐỘNG - Điều 3 0.12


In [ ]:
# from app.core.neo4j import get_driver, get_db
# article_ids = ["41/2024/QH15_D9"]  # ví dụ
# q = """
# UNWIND $article_ids AS aid
# MATCH (a:Article {article_id: aid})
# OPTIONAL MATCH (s1:Span)-[:BELONGS_TO]->(:Point  {article_id: aid})
# WITH a, aid, collect(s1)[..$limit] AS s1s
# OPTIONAL MATCH (s2:Span)-[:BELONGS_TO]->(:Clause {article_id: aid})
# WITH a, aid, s1s, collect(s2)[..$limit] AS s2s
# OPTIONAL MATCH (s3:Span)-[:BELONGS_TO]->(a)
# WITH a, aid, s1s, s2s, collect(s3)[..$limit] AS s3s
# WITH a, aid, s1s + s2s + s3s AS allSpans
# UNWIND allSpans AS s
# WITH a, aid, collect(DISTINCT s)[..$limit] AS spans
# RETURN a.article_id AS article_id, [x IN spans | {chunk_id:x.chunk_id}] AS spans
# """
# drv = get_driver()
# async with drv.session(**get_db()) as s:
#     data = await s.run(q, {"article_ids": article_ids, "limit": 20})
#     print(await data.data())

[]


In [7]:
from app.core.llm import get_chat_model
from app.agents.prompts import RAG_SYSTEM_PROMPT
context='\n'.join([c['content'] for c in ctx[:10]])
model=get_chat_model(system_instruction=RAG_SYSTEM_PROMPT+'\n'+context)
async def ans(q):
    out=''
    async for p in model.generate_stream(q): out+=p
    return out
print(await ans(query))

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash-lite\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite\nPlease retry in 56.930607138s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@